# CivicCare — Model Selection, Ensemble Learning & Comprehensive Evaluation
**Project:** Problem Statement 11 — Crowdsourced Civic Issue Reporting and Resolution Analytics Dashboard  
**Team Role:** Member 1 — Machine Learning & Data Acquisition Pipeline  
**Focus:** Comparative Model Selection, Soft-Voting Ensembling, and Holdout/Web Evaluation across all 3 Models

---

## Executive Summary
This notebook documents the comparative model selection, architectural trade-offs, and empirical validation of our **Tri-Model Municipal Intelligence Suite**:
1. **Model 1: Civic Category Classification Ensemble** (9 Master Categories) — Soft-voting ensemble combining Calibrated LinearSVC, Logistic Regression, Complement Naive Bayes, and Calibrated Ridge with word+subword FeatureUnion.
2. **Model 2: Resolution Time & SLA Compliance Engine** — Dual-head gradient-boosted engine predicting Citizen Charter SLA turnaround fulfillment and continuous hours.
3. **Model 3: Officer Response & Fake Closure Verifier** — HistGradientBoosting classifier auditing municipal officer closure remarks to detect premature brush-offs.

Every phase documents **why each algorithm was evaluated**, **why specific architectures were selected**, and evaluates models against held-out test sets as well as **50 completely new, unseen real-world citizen complaints from the web**.


In [1]:
import os
import sys
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, confusion_matrix, mean_absolute_error, median_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.autolayout': True, 'axes.titlesize': 13, 'axes.labelsize': 11})

sys.path.append(os.path.abspath('..'))
from ml.config import MASTER_CATEGORIES, NYC_CATEGORY_MAPPING, BBMP_CATEGORY_MAPPING, INDIAN_MUNICIPAL_SLAS

print("Libraries imported successfully. Ready for comparative model selection.")


Libraries imported successfully. Ready for comparative model selection.


## 1. Model 1: Comparative Selection for Category Classification

### Why evaluate multiple candidate algorithms?
Different classifiers possess distinct inductive biases:
- **Multinomial & Complement Naive Bayes:** Assume feature conditional independence given the class. Extremely fast, resilient to sparse text, but can produce overconfident probability estimates.
- **Logistic Regression:** Convex optimization maximizing log-likelihood. Produces well-calibrated posterior probabilities $\sigma(\mathbf{w}^T \mathbf{x})$.
- **Linear Support Vector Classifier (LinearSVC):** Directly optimizes the margin between classes via hinge loss. Superior boundary separation, but does not natively yield calibrated probabilities.
- **Ridge Classifier:** Minimizes squared error loss with $L2$ penalty. Strong closed-form regularizer.

We evaluate these candidates on a unified cross-city dataset to analyze empirical performance trade-offs.


In [2]:
# 1. Ingest clean cross-city descriptors
df_nyc = pd.read_parquet('../data/processed/nyc_311_2025_processed.parquet', columns=['descriptor', 'complaint_type'])
df_nyc['standard_category'] = df_nyc['complaint_type'].map(NYC_CATEGORY_MAPPING)
df_nyc = df_nyc.dropna(subset=['descriptor', 'standard_category'])
nyc_counts = df_nyc.groupby(['descriptor', 'standard_category']).size().reset_index(name='count')
nyc_desc_cat = nyc_counts.sort_values('count', ascending=False).drop_duplicates('descriptor').set_index('descriptor')['standard_category'].to_dict()

df_bbmp = pd.read_parquet('../data/processed/bbmp_grievances_processed.parquet', columns=['descriptor', 'standard_category'])
df_bbmp = df_bbmp.dropna(subset=['descriptor', 'standard_category'])
df_bbmp = df_bbmp[df_bbmp['standard_category'].isin(MASTER_CATEGORIES)]
bbmp_counts = df_bbmp.groupby(['descriptor', 'standard_category']).size().reset_index(name='count')
bbmp_desc_cat = bbmp_counts.sort_values('count', ascending=False).drop_duplicates('descriptor').set_index('descriptor')['standard_category'].to_dict()

all_desc = {}
all_desc.update(nyc_desc_cat)
all_desc.update(bbmp_desc_cat)

# Location contexts ensuring location-invariance
LOCATION_CONTEXTS = [
    'on main road', 'near ring road', 'in 4th cross lane', 'along 80 feet road',
    'near metro station', 'in our locality', 'near market circle', 'outside apartment gate'
]

rows = []
for desc, cat in all_desc.items():
    rows.append({'text': desc, 'category': cat})
    for loc in LOCATION_CONTEXTS[:4]:
        rows.append({'text': f'{desc} {loc}', 'category': cat})

df_exp = pd.DataFrame(rows)
X_exp = df_exp['text'].astype(str).tolist()
y_exp = df_exp['category'].astype(str).tolist()

X_tr_exp, X_te_exp, y_tr_exp, y_te_exp = train_test_split(X_exp, y_exp, test_size=0.25, random_state=42, stratify=y_exp)
print(f"Experimental Dataset: {len(df_exp):,} samples | Train: {len(X_tr_exp):,} | Test: {len(X_te_exp):,}")


Experimental Dataset: 3,385 samples | Train: 2,538 | Test: 847


In [3]:
feat_exp = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(ngram_range=(1, 3), max_features=25000, sublinear_tf=True)),
    ('char_tfidf', TfidfVectorizer(ngram_range=(3, 5), analyzer='char_wb', max_features=35000, sublinear_tf=True))
])

candidate_models = {
    "MultinomialNB": MultinomialNB(alpha=0.1),
    "ComplementNB": ComplementNB(alpha=0.1),
    "LogisticRegression": LogisticRegression(C=6.0, max_iter=1000, class_weight='balanced', random_state=42),
    "CalibratedLinearSVC": CalibratedClassifierCV(LinearSVC(C=1.5, class_weight='balanced', random_state=42), cv=3),
    "CalibratedRidge": CalibratedClassifierCV(RidgeClassifier(class_weight='balanced', random_state=42), cv=3)
}

print("=== CANDIDATE MODEL BENCHMARK (Individual Classifiers) ===")
bench_results = {}
for name, clf in candidate_models.items():
    pipe = Pipeline([('feats', feat_exp), ('clf', clf)])
    pipe.fit(X_tr_exp, y_tr_exp)
    preds = pipe.predict(X_te_exp)
    acc = accuracy_score(y_te_exp, preds)
    macro_f1 = f1_score(y_te_exp, preds, average='macro')
    bench_results[name] = {'Accuracy': acc, 'Macro F1': macro_f1}
    print(f"  {name:22s} -> Test Accuracy: {acc*100:.2f}% | Macro F1: {macro_f1:.4f}")


=== CANDIDATE MODEL BENCHMARK (Individual Classifiers) ===
  MultinomialNB          -> Test Accuracy: 96.58% | Macro F1: 0.9656


  ComplementNB           -> Test Accuracy: 96.46% | Macro F1: 0.9671


  LogisticRegression     -> Test Accuracy: 98.35% | Macro F1: 0.9879


  CalibratedLinearSVC    -> Test Accuracy: 98.82% | Macro F1: 0.9910


  CalibratedRidge        -> Test Accuracy: 98.35% | Macro F1: 0.9879


### Why We Chose a Soft-Voting Ensemble
**Mathematical Justification:**
Individual models can fail on ambiguous edge cases. By combining four diverse algorithms via **Soft Voting (weighted probability averaging)**:
$$\hat{y} = \arg\max_c \sum_{m=1}^{M} w_m \cdot P_m(y = c \mid \mathbf{x})$$
1. **Calibrated LinearSVC ($w=3$):** Provides sharp, maximum-margin decision boundaries.
2. **Logistic Regression ($w=3$):** Guarantees well-calibrated class log-odds.
3. **Complement Naive Bayes ($w=1$):** Strong prior handling for long-tail infrequent terms.
4. **Calibrated Ridge Classifier ($w=2$):** Regularized $L2$ stabilizer.

The weighted combination reduces estimator variance and prevents any single model's bias from dominating.


In [4]:
# Load the trained production Category Ensemble
cat_artifacts = joblib.load('../ml/models/civic_care_category_ensemble.joblib')
cat_pipeline = cat_artifacts['pipeline']

print("Loaded production Category Classification Ensemble:")
print(f"Pipeline components: {list(cat_pipeline.named_steps.keys())}")
print(f"Ensemble estimators: {[name for name, _ in cat_pipeline.named_steps['ensemble'].estimators]}")


Loaded production Category Classification Ensemble:
Pipeline components: ['feats', 'ensemble']
Ensemble estimators: ['lr', 'svc', 'cnb', 'ridge']


## 2. Model 2: Resolution Time & SLA Compliance Engine

### Why HistGradientBoosting Was Chosen
For predicting resolution turnaround times from tabular records (agency, category, borough, creation hour, day of week):
1. **LightGBM-style Integer Binning:** Bins continuous features into 256 discrete bins, reducing split evaluation from $\mathcal{O}(N \log N)$ to $\mathcal{O}(B = 256)$.
2. **OpenMP Multi-threading:** Scales across all CPU cores, training across 300,000 historical records in under 4 seconds.
3. **Non-linear Feature Interactions:** Naturally captures compound conditions (e.g., weekend night waterlogging in flood-prone boroughs).

### Dual-Head Architecture
- **Head 1 (Classifier):** Predicts binary compliance with Indian Citizen Charter SLA maximum hours.
- **Head 2 (Regressor):** Estimates continuous hours using log-transformed turnaround $\log(1 + \text{hours})$.


In [5]:
# Load Resolution Time Engine
res_artifacts = joblib.load('../ml/models/resolution_time_engine.joblib')
sla_classifier = res_artifacts['sla_classifier']
hours_regressor = res_artifacts['hours_regressor']
res_metrics = res_artifacts['metrics']

print("=== RESOLUTION TIME & SLA ENGINE BENCHMARKS (Held-out Test: 60,000 complaints) ===")
print(f"  SLA Compliance Test Accuracy : {res_metrics['accuracy']*100:.2f}% (Target: >= 80%)")
print(f"  ROC-AUC Discriminative Score : {res_metrics['roc_auc']:.4f} (Target: >= 0.85)")
print(f"  Macro F1-Score               : {res_metrics['macro_f1']:.4f}")
print(f"  Median Absolute Error        : {res_metrics['median_abs_error_hours']:.2f} hours (Resilient to tail delays)")


=== RESOLUTION TIME & SLA ENGINE BENCHMARKS (Held-out Test: 60,000 complaints) ===
  SLA Compliance Test Accuracy : 86.71% (Target: >= 80%)
  ROC-AUC Discriminative Score : 0.9262 (Target: >= 0.85)
  Macro F1-Score               : 0.8048
  Median Absolute Error        : 6.87 hours (Resilient to tail delays)


## 3. Model 3: Officer Response Quality & Fake Closure Verifier

### Why an Anomaly Detection Model on Officer Remarks?
To solve Problem Statement 11's requirement to flag premature closures:
Field officers facing resolution targets frequently write single-character or canned brush-offs (e.g. `"."`, `"-"`, `"closed"`, `"Action will be taken"`).

### Architecture
- Model: `HistGradientBoostingClassifier(class_weight='balanced')` trained on 1.2M BBMP records.
- Features: remark length, word count, canned phrase flags, future promise flags, and prior channel closure rates.


In [6]:
# Load Officer Response Engine
officer_artifacts = joblib.load('../ml/models/officer_response_engine.joblib')
officer_model = officer_artifacts['model']
officer_metrics = officer_artifacts['metrics']

print("=== OFFICER RESPONSE QUALITY VERIFIER (Held-out Test: 241,550 complaints) ===")
print(f"  Verification Accuracy : {officer_metrics['accuracy']*100:.2f}% (Target: >= 80%)")
print(f"  ROC-AUC Score         : {officer_metrics['roc_auc']:.4f} (Target: >= 0.85)")
print(f"  Macro F1-Score        : {officer_metrics['macro_f1']:.4f}")
print(f"  Weighted F1-Score     : {officer_metrics['weighted_f1']:.4f}")


=== OFFICER RESPONSE QUALITY VERIFIER (Held-out Test: 241,550 complaints) ===
  Verification Accuracy : 92.95% (Target: >= 80%)
  ROC-AUC Score         : 0.9634 (Target: >= 0.85)
  Macro F1-Score        : 0.8355
  Weighted F1-Score     : 0.9350


## 4. Independent Validation on 50 Real Web Complaints

### The Ultimate Test of Generalization
To prove that our models are **neither overfitting to training templates nor underfitting real-world phrasing**, we evaluate against **50 completely new, unseen citizen complaints sourced from Indian municipal grievance forums and social media (X/Twitter, CPGRAMS, Reddit r/bangalore)** across all 9 categories.


In [7]:
from ml.verify_new_web_data import run_web_verification

run_web_verification()


INDEPENDENT GENERALIZATION, LEAKAGE & OVERFITTING VERIFICATION ON WEB DATA

[1/4] Testing Category Classification Ensemble on 50 Real Web Complaints...



>>> RESULTS ON 100% UNSEEN REAL-WORLD WEB DATA <<<
Overall Accuracy:        98.00% (Target: >= 80%)
Macro F1-Score:          0.9798 (Target: >= 0.80)
Weighted F1-Score:       0.9800
Average Model Confidence: 93.4%
Total Test Cases:        50
Correct Predictions:     49 / 50

Mismatches (1 total):
  - 'Litter bins overflowing near vegetable market with stray cattle and fl...'
    Expected: Solid Waste Management | Predicted: Animal Husbandry & Stray Animals (Confidence: 83.8%)

Detailed Per-Class Performance on Web Data:
                                  precision    recall  f1-score   support

Animal Husbandry & Stray Animals     0.8333    1.0000    0.9091         5
         Drainage & Water Supply     1.0000    1.0000    1.0000         6
       Encroachments & Footpaths     1.0000    1.0000    1.0000         5
         Horticulture & Greenery     1.0000    1.0000    1.0000         5
      Public Health & Sanitation     1.0000    1.0000    1.0000         6
      Revenue & Urban Govern

  Remark: '-                                            ' -> Risk:  99.8% | Pred: FLAGGED (Fake Closure)  | Expected: Fake Closure [PASS]
  Remark: 'close                                        ' -> Risk:  98.0% | Pred: FLAGGED (Fake Closure)  | Expected: Fake Closure [PASS]
  Remark: 'attended                                     ' -> Risk:   7.5% | Pred: VERIFIED (Genuine)      | Expected: Fake Closure [FAIL]
  Remark: 'Action will be taken shortly                 ' -> Risk:  87.7% | Pred: FLAGGED (Fake Closure)  | Expected: Fake Closure [PASS]
  Remark: '1st Assignment Based on Ward Mapping         ' -> Risk:   5.3% | Pred: VERIFIED (Genuine)      | Expected: Fake Closure [FAIL]
  Remark: 'Forwarded task to junior engineer for further action' -> Risk:  90.2% | Pred: FLAGGED (Fake Closure)  | Expected: Fake Closure [PASS]

Officer Response Verifier Accuracy on Web Remarks: 83.33% (10/12)

VERIFICATION COMPLETED SUCCESSFULLY!


## 5. End-to-End Live Prediction Interface
Demonstrating the unified tri-model pipeline consumed by the C++ application (Member 3 & Member 4) and Web Dashboard.


In [8]:
from ml.predict import predict_civic_complaint

test_cases = [
    ("Severe pothole on main road causing vehicle damage and traffic bottleneck", ""),
    ("Streetlight pole sparking and blacked out near high school junction", "attended"),
    ("Open garbage dump and uncollected blackspot creating severe stench", "."),
    ("Stormwater drain clogged with plastic waste causing road waterlogging", "Action will be taken shortly")
]

print("=== REAL-TIME TRI-MODEL PREDICTION CONTRACT ===")
for text, rem in test_cases:
    res = predict_civic_complaint(text, remarks=rem)
    print(f"\nComplaint Description: '{text}'")
    if rem:
        print(f"Officer Closure Remark: '{rem}'")
    print(f"  -> Predicted Category:   {res['category']} (Confidence: {res['confidence']*100:.1f}%)")
    print(f"  -> Estimated Turnaround: {res['estimated_resolution_hours']} hrs ({res['estimated_resolution_days']} days)")
    print(f"  -> SLA Delivery Tier:    {res['sla_tier']} (Compliance Prob: {res['sla_compliance_probability']*100:.1f}%)")
    print(f"  -> Recommended Priority: {res['recommended_priority']}")
    print(f"  -> Officer Action:       {res['officer_response']['recommended_action']}")
    print(f"  -> Response Integrity:   {res['officer_response']['verification_status']} (Anomaly Risk: {res['officer_response']['fake_closure_risk']*100:.1f}%)")


=== REAL-TIME TRI-MODEL PREDICTION CONTRACT ===

Complaint Description: 'Severe pothole on main road causing vehicle damage and traffic bottleneck'
  -> Predicted Category:   Roads & Infrastructure (Confidence: 83.4%)
  -> Estimated Turnaround: 54.5 hrs (2.27 days)
  -> SLA Delivery Tier:    Standard (24 to 72 Hours) (Compliance Prob: 48.2%)
  -> Recommended Priority: Critical
  -> Officer Action:       Field Crew: Road Maintenance & Asphalt Paving
  -> Response Integrity:   Not Applicable (Ticket Open) (Anomaly Risk: 5.0%)



Complaint Description: 'Streetlight pole sparking and blacked out near high school junction'
Officer Closure Remark: 'attended'
  -> Predicted Category:   Street Lighting & Electrical (Confidence: 99.7%)
  -> Estimated Turnaround: 28.6 hrs (1.19 days)
  -> SLA Delivery Tier:    Immediate (Within 24 Hours) (Compliance Prob: 40.1%)
  -> Recommended Priority: Critical
  -> Officer Action:       Field Crew: Electrical Lineman & Luminaire Replacement
  -> Response Integrity:   Verified: Genuine Resolution (Anomaly Risk: 2.2%)

Complaint Description: 'Open garbage dump and uncollected blackspot creating severe stench'
Officer Closure Remark: '.'
  -> Predicted Category:   Solid Waste Management (Confidence: 97.8%)
  -> Estimated Turnaround: 26.3 hrs (1.1 days)
  -> SLA Delivery Tier:    Immediate (Within 24 Hours) (Compliance Prob: 75.4%)
  -> Recommended Priority: Critical
  -> Officer Action:       Sanitation Squad: Compact Truck & Debris Clearing
  -> Response Integrity:   Flagged: Prema